# 第 5 讲实验：GPU / TPU 与 Roofline

本 Notebook 把理论估算、Roofline 上界和实际 benchmark 分开。Roofline 示例使用人为设定的教学参数，不代表任何具体 GPU 的规格。benchmark 会自动优先使用 CUDA；无 CUDA 时使用 CPU fallback，并明确不将其解释为 GPU 性能。

In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

START = Path.cwd().resolve()
REPO_ROOT = next((p for p in (START, *START.parents) if (p / "code" / "lecture_05").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("请在课程仓库内启动 Jupyter Lab")
sys.path.insert(0, str(REPO_ROOT / "code" / "lecture_05"))
from accelerator_lab import (
    tensor_bytes, matmul_flops, matmul_ideal_bytes,
    arithmetic_intensity, roofline_bound, benchmark_matmul,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("device:", device)
if device.type == "cpu":
    print("CPU fallback：可验证公式和 benchmark 代码路径，但不代表 GPU 性能。")

## 实验 1：FLOPs、字节数与算术强度

对矩阵乘 C=A×B，A 的形状是 [M,K]，B 是 [K,N]。按乘加计 2 FLOPs；理想化字节模型假设 A/B 各读一次、C 写一次。实际 HBM traffic 需要 profiler 验证。

In [ ]:
M, K, N = 1024, 2048, 512
dtype = torch.float32
flops = matmul_flops(M, N, K)
ideal_bytes = matmul_ideal_bytes(M, N, K, dtype)
intensity = arithmetic_intensity(flops, ideal_bytes)
print(f"A [{M},{K}] × B [{K},{N}]")
print(f"FLOPs (multiply-add = 2): {flops:,}")
print(f"A bytes: {tensor_bytes((M, K), dtype):,}")
print(f"B bytes: {tensor_bytes((K, N), dtype):,}")
print(f"C bytes: {tensor_bytes((M, N), dtype):,}")
print(f"idealized total bytes: {ideal_bytes:,} ({ideal_bytes / 1024**2:.2f} MiB)")
print(f"idealized arithmetic intensity: {intensity:.2f} FLOPs/byte")
assert ideal_bytes == tensor_bytes((M, K), dtype) + tensor_bytes((K, N), dtype) + tensor_bytes((M, N), dtype)

## 实验 2：绘制教学用 Roofline 曲线

这里的 100 TFLOP/s 和 2 TB/s 是人为设置的示例参数，只用于观察 ridge point，不是某款 GPU 的规格。实际分析时必须从目标设备资料和 profiler 获取数值。

In [ ]:
import numpy as np

example_peak_flops = 100e12
example_bandwidth = 2e12
intensities = np.logspace(-2, 3, 300)
bounds = [roofline_bound(example_peak_flops, example_bandwidth, value) for value in intensities]
ridge_point = example_peak_flops / example_bandwidth
plt.figure(figsize=(7, 4))
plt.loglog(intensities, bounds, label="Roofline upper bound")
plt.axvline(ridge_point, linestyle="--", label=f"Ridge point ≈ {ridge_point:.0f} FLOPs/byte")
plt.xlabel("Arithmetic intensity (FLOPs/byte)")
plt.ylabel("Performance upper bound (FLOPs/s)")
plt.title("Illustrative Roofline (synthetic parameters, not device specs)")
plt.grid(True, which="both", alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()
print("ridge point =", ridge_point, "FLOPs/byte")

## 实验 3：同步正确的 matmul smoke benchmark

CUDA 路径使用 CUDA events 和同步；CPU 路径使用 perf_counter。这个实验验证 benchmark 流程与误差计算，不应只凭一次结果作硬件排名。

In [ ]:
results = []
dtypes = [torch.float32]
if device.type == "cuda" and torch.cuda.is_bf16_supported():
    dtypes.append(torch.bfloat16)

for dtype in dtypes:
    result = benchmark_matmul(
        256, 256, 256, dtype=dtype, device=device,
        warmup=2, iterations=5, seed=42,
    )
    results.append(result)
    print(
        f"dtype={result['dtype']}, latency={result['latency_ms']:.3f} ms, "
        f"relative RMSE={result['relative_rmse_vs_fp32']:.3e}, "
        f"intensity={result['idealized_arithmetic_intensity_flops_per_byte']:.2f} FLOPs/byte"
    )
if device.type == "cpu":
    print("本次只测 CPU FP32 smoke path；没有 GPU 性能数据。")
assert all(r["latency_ms"] > 0 for r in results)
assert all(torch.isfinite(torch.tensor(r["relative_rmse_vs_fp32"])).item() for r in results)

## 实验 4：查看延迟和数值误差

若当前设备只运行一种 dtype，图中只有一个柱子；这不是跨 dtype 的性能比较。GPU BF16 结果只在设备支持时加入。

In [ ]:
labels = [r["dtype"].replace("torch.", "") for r in results]
plt.figure(figsize=(6, 3.5))
plt.bar(labels, [r["latency_ms"] for r in results])
plt.ylabel("Mean latency (ms)")
plt.title(f"Matmul smoke latency on {device}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3.5))
plt.bar(labels, [r["relative_rmse_vs_fp32"] for r in results])
plt.ylabel("Relative RMSE vs FP32 reference")
plt.title("Numerical error (lower is closer to FP32 reference)")
plt.tight_layout()
plt.show()

## 复盘

1. 为什么 Roofline 给的是上界，而不是精确延迟？
2. 理想化 bytes 模型忽略了哪些 cache、tiling 与访存行为？
3. 为什么高 occupancy 不一定等于高性能？
4. 哪些结果是理论估算，哪些来自当前设备的真实测量？
5. 如果要比较 FP32 与 BF16，为什么必须同时报告误差、dtype、设备、warm-up、同步和重复次数？